In [2]:
import os
import pandas as pd
import pingouin as pg
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tools.draw import *
from sklearn.preprocessing import StandardScaler
from scipy.stats import shapiro, kruskal

###### 1代表男，2代表女
###### 类型：1普通，2吸烟
###### 背景：1中性，2厌恶

In [3]:
data = pd.read_excel('datum\map_data.xlsx')
data

,name,age,sex,smoke_year,smoke_quantify,FTND,emotion,pre,ing,post,...,punish,unpunish,common rewarded,rare rewarded,common unrewarded,rare unrewarded,common unpunished,rare unpunished,common punished,rare punished
0,蔡,21,1,0.0,0.0,0,70,0,0,0,...,0.07,0.00,1.00,1.00,0.86,1.00,1.00,1.00,0.93,1.00
1,戴,18,2,0.0,0.0,0,80,0,0,0,...,0.02,0.00,1.00,0.67,0.01,1.00,1.00,0.00,0.98,0.00
2,丁,19,2,0.0,0.0,0,80,0,0,0,...,0.82,-0.44,0.91,0.26,0.06,0.80,0.96,0.43,0.14,0.87
3,葛,23,2,0.0,0.0,0,60,0,0,0,...,0.92,0.67,1.00,1.00,0.04,0.00,0.95,0.82,0.03,0.15
4,何,20,1,0.0,0.0,0,80,0,0,0,...,0.64,-0.42,1.00,0.00,0.14,1.00,1.00,0.50,0.36,0.92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,迷糊,22,1,4.0,6.0,2,60,60,70,50,...,0.70,-0.02,0.94,0.50,0.13,0.90,0.94,0.55,0.24,0.57
158,勿忘我,22,1,3.0,5.0,4,85,70,85,100,...,0.93,-0.52,1.00,0.00,0.03,0.88,0.93,0.25,0.00,0.77
159,徐桑,22,1,2.0,4.0,3,60,10,30,50,...,1.00,-1.00,0.98,0.09,0.00,1.00,1.00,0.00,0.00,1.00
160,俞启越,23,1,3.0,6.0,3,75,20,55,55,...,0.98,-1.00,0.97,0.00,0.04,1.00,0.98,0.00,0.00,1.00


##### age/sex/number of each group

In [ ]:
filtered_data  = data[(data['type'] == 2) & (data['background'] ==2)]
var = filtered_data['smoke_quantify']
(var != 'nan').sum()
print((np.mean(var)).round(2),(np.std(var)).round(2))

##### ANOVA of craving (induce)

In [ ]:
## prepare dataframe
score = pd.concat((data['pre'], data['ing']))
raw_group = np.tile(data['type'], 2)
group = ['ND' if raw_group[t] == 2 else 'HC' for t in range(len(raw_group))]
time = ['before']*len(data) + ['after']*len(data)
subject = ([f'subject{i}' for i in range(1, len(data)+1)]) * 2
new_df = pd.DataFrame({'value':score, 'group':group, 'time':time, 'subject':subject})

In [ ]:
pg.mixed_anova(data = new_df, dv = 'value', subject='subject',
               between = 'group', within = 'time')


In [ ]:
fig, ax = plt.subplots(figsize=(2, 1.8), dpi=300)
font = {'family': 'Arial', 'weight': 'regular'}
plt.rc('font', **font)
color = {'before': '#c1bdb1', 'after': '#3e5751'}
y_name = 'craving score'
 
violin(ax, data = new_df, x = 'group', y = 'value',
       hue = 'time', order = ['HC', 'ND'], 
       hue_order = ['before', 'after'], palette = color)
ax.get_legend().remove()
basic_format(ax, '', f'{y_name}')
plt.show()

In [ ]:
df2 = new_df[new_df['group'] == 'ND']
before = df2[df2['time'] == 'before']['value']
after = df2[df2['time'] == 'after']['value']
pg.ttest(before, after, paired = True)

##### ANOVA of craving /best_choice(decrease)

In [ ]:
filtered_data = data[(data['type'] == 2) & (data['background'] == 1)]
value1 = filtered_data['ing']
value2 = filtered_data['post']
pg.ttest(value1, value2,  paired = True)

In [ ]:
## prepare dataframe
filtered_data = data[data['type'] == 2]

score = pd.concat((filtered_data['ing'], filtered_data['post']))

raw_group = np.tile(filtered_data['background'], 2)
group = ['neutral' if raw_group[t] == 1 else 'aversive' for t in range(len(raw_group))]
time = ['before']*len(filtered_data) + ['after']*len(filtered_data)
subject = ([f'subject{i}' for i in range(1, len(filtered_data)+1)]) * 2
new_df = pd.DataFrame({'value':score, 'group':group, 'time':time, 'subject':subject})


In [ ]:
pg.mixed_anova(data = new_df, dv = 'value', subject='subject',
               between = 'group', within = 'time')

In [ ]:
fig, ax = plt.subplots(figsize=(2, 1.8), dpi=300)
font = {'family': 'Arial', 'weight': 'regular'}
plt.rc('font', **font)
color = {'before': '#c1bdb1', 'after': '#3e5751'}
y_name = 'craving score'
 
violin(ax, data = new_df, x = 'group', y = 'value',
       hue = 'time', order = ['neutral', 'aversive'], 
       hue_order = ['before', 'after'], palette = color)
ax.get_legend().remove()
basic_format(ax, '', f'{y_name}')
plt.show()

##### ANOVA of emotion /best_choice(induce)

In [ ]:
## prepare dataframe
param = 'best choice rate_pun'
score = np.log(data[param])
raw_group = data['type']
group = ['ND' if raw_group[t] == 2 else 'HC' for t in range(len(raw_group))]
raw_time = data['background']
time =  ['aversive' if raw_time[t] == 2 else 'neutral' for t in range(len(raw_time))]
subject = ([f'subject{i}' for i in range(1, len(data)+1)])
new_df = pd.DataFrame({'value':score, 'group':group, 'time':time, 'subject':subject})
new_df

In [ ]:
pg.anova(data = new_df, dv = 'value', between = ['group', 'time'])

In [ ]:
df2 = new_df[new_df['group'] == 'HC']
before = df2[df2['time'] == 'neutral']['value']
after = df2[df2['time'] == 'aversive']['value']
pg.ttest(before, after)

In [ ]:
fig, ax = plt.subplots(figsize=(2, 1.8), dpi=300)
font = {'family': 'Arial', 'weight': 'regular'}
plt.rc('font', **font)
color = {'neutral': [23/255, 66/255, 99/255], 'aversive':  [215/255, 87/255, 40/255]}
y_name = param
 
violin(ax, data = new_df, x = 'group', y = 'value',
       hue = 'time', order = ['HC', 'ND'], 
       hue_order = ['neutral', 'aversive'], palette = color)
ax.get_legend().remove()
basic_format(ax, '', f'{y_name}')
plt.show()

    * 类型：1普通，2吸烟
    * 背景：1中性，2厌恶

##### ANOVA of w in HC/ND

In [ ]:
# Ensure proper indexing with .iloc
param = 'best choice rate'
use_name = 'HC'
use_value = 1 if use_name == 'HC' else 2
filtered_data = data[data['type'] == use_value]
score = np.log(pd.concat((filtered_data[f'{param}_rew'], filtered_data[f'{param}_pun'])))
raw_group = filtered_data['background']

# Using .iloc to access by position and fixing group assignment
group = ['aversive' if raw_group.iloc[t] == 2 else 'neutral' for t in range(len(filtered_data))] * 2

# Ensure 'time' and 'subject' lists are of appropriate length
time = ['reward'] * len(filtered_data) + ['punishment'] * len(filtered_data)
subject = [f'subject{i}' for i in range(1, len(filtered_data) + 1)] * 2

new_df = pd.DataFrame({'value': score, 'group': group, 'time': time, 'subject': subject})


In [ ]:
pg.mixed_anova(data = new_df, dv = 'value', between = 'group', 
               within = 'time', subject = 'subject')

In [ ]:
df2 = new_df[new_df['time'] == 'reward']
before = df2[df2['group'] == 'neutral']['value']
after = df2[df2['group'] == 'aversive']['value']
pg.ttest(before, after)

In [ ]:
fig, ax = plt.subplots(figsize=(2, 1.8), dpi=300)
font = {'family': 'Arial', 'weight': 'regular'}
plt.rc('font', **font)
color = {'neutral': [23/255, 66/255, 99/255], 'aversive':  [215/255, 87/255, 40/255]}
y_name = param

violin(ax, data = new_df, x = 'time', y = 'value',
       hue = 'group', order = ['reward', 'punishment'], 
       hue_order = ['neutral', 'aversive'], palette = color)
ax.get_legend().remove()
basic_format(ax, '', f'{y_name}')
save_path = f'config_filter'
plt.show()

##### linear regression

In [ ]:
## prepare dataframe
## smoke_quantify, smoke_year
filtered_data = data[(data['background'] == 2) & (data['type'] == 2)]
y = np.log(filtered_data['alpha1_pun'])
x = filtered_data['smoke_year']

pg.linear_regression(X = x, y = y, add_intercept = True,remove_na=True)

In [ ]:
fig, ax = plt.subplots(figsize = (2, 1.5), dpi = 300)
sns.regplot(
            x=x, y=y, color="indianred",
            line_kws={'color': 'indianred', 'alpha': 0.3, 'zorder': 1},
            scatter_kws={'zorder': 1} 
            )
basic_format(ax, 'Cigarettes per day',r'$\alpha_1$')
